# Component 2 - Demand, Spam, and Video Context Annotation Preparation

This notebook prepares a deterministic manual annotation sample for audience-demand labels, separate human spam-status labels, and read-only video context for annotators. It does not train a classifier, run BERTopic, calculate Jensen-Shannon Divergence, collect new YouTube data, or connect to Supabase.

Visible notebook outputs are limited to aggregate checks and distribution tables; raw comment text is written only to the local annotation CSV used for manual labelling.

In [1]:
from pathlib import Path
import sys

import pandas as pd

start = Path.cwd().resolve()
for candidate in [start, *start.parents]:
    if (candidate / "research" / "component_2" / "src" / "annotation.py").exists():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate repository root from notebook working directory")

SRC = REPO_ROOT / "research" / "component_2" / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from annotation import (
    ANNOTATION_COLUMNS,
    HUMAN_ANNOTATION_COLUMNS,
    SPAM_ANNOTATION_COLUMNS,
    VIDEO_CONTEXT_COLUMNS,
    AnnotationSamplingConfig,
    add_spam_annotation_fields,
    add_video_context_fields,
    create_annotation_sample,
    create_annotation_template,
    nonblank_annotation_counts,
    prepare_annotation_source,
    validate_existing_annotation_template,
    write_annotation_outputs,
)

RAW_DIR = REPO_ROOT / "data" / "raw" / "component_2"
PROCESSED_DIR = REPO_ROOT / "data" / "processed" / "component_2"
ANNOTATION_DIR = PROCESSED_DIR / "annotation"
ANNOTATION_CSV = ANNOTATION_DIR / "component2_demand_annotation_sample.csv"

preprocessed_path = PROCESSED_DIR / "preprocessed_comments.csv"
validated_path = PROCESSED_DIR / "validated_comments.csv"
raw_videos_path = RAW_DIR / "component2_pilot_videos.csv"
raw_comments_path = RAW_DIR / "component2_pilot_comments.csv"

for required_path in [preprocessed_path, validated_path, raw_videos_path, raw_comments_path]:
    assert required_path.exists(), f"Missing {required_path}"

## Load Phase 1 Outputs and Pilot Context Metadata

The annotation source is built from Phase 1 preprocessed comments plus the validated comment metadata needed for channel names, reply status, and publication time. Video titles and descriptions come from the raw pilot video export using `video_id`. Parent-comment text is resolved only when the parent comment is present in the raw pilot comment export.

In [2]:
preprocessed_comments = pd.read_csv(
    preprocessed_path,
    dtype={"comment_id": "string", "channel_id": "string", "video_id": "string"},
)
validated_comments = pd.read_csv(
    validated_path,
    dtype={"comment_id": "string", "parent_comment_id": "string"},
)
pilot_videos = pd.read_csv(
    raw_videos_path,
    dtype={"video_id": "string", "title": "string", "description": "string"},
)
pilot_comments = pd.read_csv(
    raw_comments_path,
    dtype={"comment_id": "string", "parent_comment_id": "string", "comment_text": "string"},
)

source = prepare_annotation_source(preprocessed_comments, validated_comments)

source_summary = pd.DataFrame(
    [
        {
            "source_comments": len(source),
            "source_channels": source["channel_id"].nunique(),
            "source_videos": source["video_id"].nunique(),
            "pilot_video_rows": len(pilot_videos),
            "pilot_comment_rows": len(pilot_comments),
            "reply_comments": int(source["is_reply"].sum()),
            "empty_cleaned_text": int(source["cleaned_text"].fillna("").str.strip().eq("").sum()),
        }
    ]
)
source_summary

,source_comments,source_channels,source_videos,pilot_video_rows,pilot_comment_rows,reply_comments,empty_cleaned_text
0,2000,5,255,327,2000,367,0


## Create or Safely Enrich Deterministic Annotation Sample

The target sample size remains 300 comments. The sampler balances channels first, then deliberately includes minority and review-sensitive cases such as mixed-script comments, emoji-only comments, replies, very short comments, Sinhala-script comments, and Latin-script comments. Selection order is deterministic using a stable hash of `comment_id` and the configured seed.

If the annotation CSV already exists, this notebook validates that the same comment rows and ordering are preserved. Existing human demand or spam annotations are preserved while video context columns are added or refreshed.

In [3]:
config = AnnotationSamplingConfig(sample_size=300, random_seed=334)
sample = create_annotation_sample(source, config)
fresh_template = create_annotation_template(sample)

if ANNOTATION_CSV.exists():
    existing_template = pd.read_csv(ANNOTATION_CSV, dtype="string").fillna("")
    validate_existing_annotation_template(existing_template, fresh_template)
    template = add_spam_annotation_fields(existing_template)
    template_action = "enriched_existing_csv"
else:
    template = fresh_template
    template_action = "created_new_csv"

template = add_video_context_fields(template, pilot_videos, pilot_comments)
paths = write_annotation_outputs(template, ANNOTATION_DIR)

annotation_counts = nonblank_annotation_counts(template)
checks = pd.DataFrame(
    [
        {
            "template_action": template_action,
            "annotation_rows": len(template),
            "unique_comment_ids": template["comment_id"].nunique(),
            "channels_covered": template["channel_id"].nunique(),
            "videos_covered": template["video_id"].nunique(),
            "matched_video_titles": int(template["video_title"].fillna("").str.strip().ne("").sum()),
            "missing_video_titles": int(template["video_title"].fillna("").str.strip().eq("").sum()),
            "missing_video_descriptions": int(template["video_description"].fillna("").str.strip().eq("").sum()),
            "resolved_parent_comments": int(template["parent_comment_text"].fillna("").str.strip().ne("").sum()),
            "blank_spam_status_fields": int(
                sum(template[column].fillna("").eq("").sum() for column in SPAM_ANNOTATION_COLUMNS)
            ),
            "human_annotation_nonblank_total": sum(annotation_counts.values()),
            "schema_matches_expected": template.columns.tolist() == ANNOTATION_COLUMNS,
        }
    ]
)
checks

,template_action,annotation_rows,unique_comment_ids,channels_covered,videos_covered,matched_video_titles,missing_video_titles,missing_video_descriptions,resolved_parent_comments,blank_spam_status_fields,human_annotation_nonblank_total,schema_matches_expected
0,enriched_existing_csv,300,300,5,152,300,0,0,23,1200,0,True


## Sample Distribution Checks

These aggregate tables verify coverage without displaying raw comment text.

In [4]:
by_channel = pd.read_csv(paths["by_channel"])
by_script_category = pd.read_csv(paths["by_script_category"])
by_sampling_stratum = pd.read_csv(paths["by_sampling_stratum"])
by_reply_status = pd.read_csv(paths["by_reply_status"])

by_channel

,channel_id,channel_name,comments,videos
0,UCCK3OZi788Ok44K97WAhLKQ,Ada Derana,60,39
1,UCHKtXGbqRIM0hA4Manb9DNg,Ada Derana Sports,60,27
2,UCOPPH1_whpThgY6e7r3qg-w,FM Derana,60,25
3,UCRDDfbYPHX_GUJ4lcQYTc8A,TV Derana,60,26
4,UCs8gVM6lPhRZk3L_zZmedpw,Dream Star,60,35


In [5]:
by_script_category

,script_category,count
0,latin_script,106
1,sinhala_script,91
2,emoji_only,61
3,mixed_sinhala_latin,40
4,other_or_symbolic,2


In [6]:
by_sampling_stratum

,sampling_stratum,count
0,sinhala_script,74
1,latin_script,68
2,emoji_only,61
3,reply,43
4,mixed_sinhala_latin,40
5,very_short,14


In [7]:
by_reply_status

,is_reply,count
0,False,234
1,True,66


## Output Files

The generated annotation CSV contains public comment text and video context, and should stay local unless the team explicitly approves a sharing path. It is ignored by Git under `data/processed/component_2/annotation/`. The video context fields are read-only helper fields for human interpretation; demand and spam labels remain separate human annotation decisions.

In [8]:
pd.DataFrame(
    [{"artifact": name, "path": str(path.relative_to(REPO_ROOT))} for name, path in paths.items()]
)

,artifact,path
0,annotation_csv,data/processed/component_2/annotation/componen...
1,by_channel,data/processed/component_2/annotation/by_chann...
2,by_script_category,data/processed/component_2/annotation/by_scrip...
3,by_sampling_stratum,data/processed/component_2/annotation/by_sampl...
4,by_reply_status,data/processed/component_2/annotation/by_reply...


## Manual Labelling Instructions

Use `research/component_2/annotation_guidelines.md` while filling the blank demand and spam annotation fields. Read the original comment first, then use the video title, description, URL, and parent comment text when needed to resolve ambiguity. Do not create synthetic labels. Demand classification and spam classification are separate decisions. After human annotation is complete, the next research step is inter-annotator/adjudication review before any classifier, topic modelling, automated spam filtering, or demand-supply divergence work begins.